# Preparar el dataset grande en Colab

Este notebook **no es el entregable**: es una herramienta para pasar el CSV de
~8 GB (guardado en Drive) a Parquet (~2.3 GB) y generar una muestra manejable
para el notebook del proyecto.

## Pasos
1. Subir el CSV a Drive (por ejemplo, a `Mi unidad/ED2/transferencias.csv`).
2. Ejecutar las celdas en orden.
3. Al final tendrás:
   - `transferencias.parquet` (~2.3 GB) para consultar con DuckDB.
   - `data/muestra/transferencias_100k.csv` para el notebook del proyecto.


## 1. Montar Google Drive

Autoriza el acceso cuando Colab lo pida.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive montado en /content/drive")

## 2. Clonar el repositorio e instalar dependencias

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/arcibyte/ethereum-exchange"
REPO_DIR = "ethereum-exchange"

# Evita clonar dos veces si la celda se ejecuta de nuevo.
if os.path.basename(os.getcwd()) != REPO_DIR:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-datos.txt"],
    check=True,
)
sys.path.insert(0, os.getcwd())
print("Repositorio listo en:", os.getcwd())

## 3. Configurar rutas

Cambia `RUTA_CSV` por la ruta real del archivo en tu Drive.

- El Parquet se escribe en el **disco local de Colab** (`/content/`), que es
  más rápido. Si quieres conservarlo entre sesiones, copia al final la línea
  comentada que lo pasa a Drive.
- La muestra se escribe dentro del repo, en `data/muestra/`.

In [ ]:
import os

# <-- CAMBIA ESTA RUTA por la de tu CSV en Drive
RUTA_CSV = "/content/drive/MyDrive/ED2/transferencias.csv"

RUTA_PARQUET = "/content/transferencias.parquet"
RUTA_MUESTRA = "/content/ethereum-exchange/data/muestra/transferencias_100k.csv"

if not os.path.isfile(RUTA_CSV):
    raise SystemExit(f"No encuentro el CSV en: {RUTA_CSV}\nRevisa la ruta en tu Drive.")

tamano_gb = os.path.getsize(RUTA_CSV) / (1024 ** 3)
print(f"CSV encontrado: {RUTA_CSV} ({tamano_gb:.2f} GB)")

## 4. Convertir el CSV a Parquet

Esto puede tardar varios minutos. La salida esperada es un archivo de ~2.3 GB.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "scripts/preparar_dataset.py", "convertir",
     "--entrada", RUTA_CSV, "--salida", RUTA_PARQUET],
    check=True,
)

## 5. Ver estadísticas del Parquet

Comprueba cuántas transferencias y direcciones distintas quedaron.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "scripts/preparar_dataset.py", "info", "--entrada", RUTA_PARQUET],
    check=True,
)

## 6. Generar la muestra para el notebook

`--filas` controla el tamaño. Para los algoritmos conviene **V ≤ 2000 y
E ≤ 5000**, así que empieza con 100 000 filas y baja si el notebook va lento.

`--componente` conserva solo la red conectada más grande.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "scripts/preparar_dataset.py", "muestra",
     "--entrada", RUTA_PARQUET,
     "--salida", RUTA_MUESTRA,
     "--filas", "100000",
     "--componente"],
    check=True,
)
print("Muestra lista en:", RUTA_MUESTRA)

## 7. (Opcional) Guardar el Parquet en Drive

Si quieres reutilizar el Parquet en otra sesión sin volver a convertir el CSV,
cópialo a Drive. Ojo con el espacio: el CSV (8 GB) + el Parquet (2.3 GB) deben
caber en tu cuota.

In [ ]:
# import shutil
# shutil.copy(RUTA_PARQUET, "/content/drive/MyDrive/ED2/transferencias.parquet")
# print("Parquet copiado a Drive.")

# También puedes descargar la muestra a tu computador:
# from google.colab import files
# files.download(RUTA_MUESTRA)

## Siguiente paso

En el notebook del proyecto, cambia la ruta de la muestra:

```python
RUTA_TRANSFERENCIAS = "data/muestra/transferencias_100k.csv"
```

Si el notebook se vuelve lento, reduce `--filas` (por ejemplo, a 20 000) y
vuelve a generar la muestra.
